# CZU-MHAD Deep Learning Pipeline

In [1]:
import os
import numpy as np
import pandas as pd
import random
import scipy.io as sio
from scipy.io import loadmat
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
csv_filename = "records.csv"
csv_path = os.path.join(parent_dir, csv_filename)

BASE_DIR = "CZU-MHAD"
DATASET_DIR = os.path.join(BASE_DIR, "CZU-MHAD")

## CZU-MHAD Dataset Loading

In [2]:
def load_czu_mhad_dataset():
    SUBFOLDERS = ["depth_mat", "sensor_mat", "skeleton_mat"]

    all_files = os.listdir(os.path.join(DATASET_DIR, "sensor_mat"))
    subjects = sorted(list({f.split("_")[0] for f in all_files if f.endswith(".mat")}))
    print("Subjects:", subjects)

    all_samples = []
    for file in os.listdir(os.path.join(DATASET_DIR, "sensor_mat")):
        if not file.endswith(".mat"):
            continue

        subj = file.split("_")[0]
        base_name = file.replace(".mat", "")
        label = file.split("_")[1]

        # Check if corresponding files exist in all modalities
        exists_in_all = all(
            os.path.exists(os.path.join(DATASET_DIR, sub, base_name + ".mat"))
            for sub in SUBFOLDERS
        )

        if not exists_in_all:
            print(f"Skipping incomplete sample: {base_name}")
            continue

        all_samples.append({"subject": subj, "label": label, "basename": base_name})

    print(f"Total samples loaded metadata: {len(all_samples)}")


    def load_sample(sample):
        data = {}
        for sub in SUBFOLDERS:
            fpath = os.path.join(DATASET_DIR, sub, sample["basename"] + ".mat")
            try:
                mat = loadmat(fpath)
                key = [k for k in mat.keys() if not k.startswith("__")][0]
                data[sub] = mat[key]
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
                return None

        data["label"] = sample["label"]
        data["subject"] = sample["subject"]
        data["basename"] = sample["basename"]
        return data

    all_data = []
    for s in tqdm(all_samples, desc="Loading all samples"):
        d = load_sample(s)
        if d is not None:
            all_data.append(d)

    print(f"Total samples loaded: {len(all_data)}")
    return all_data


X = load_czu_mhad_dataset()

Subjects: ['cx', 'cyy', 'myj', 'qyh', 'zyh']
Total samples loaded metadata: 1165


Loading all samples: 100%|██████████| 1165/1165 [01:02<00:00, 18.65it/s]

Total samples loaded: 1165


In [3]:
sample = X[0]
print("\nSample keys:", sample.keys())
print("Depth shape:", sample["depth_mat"].shape)
print("Sensor shape:", sample["sensor_mat"].shape)
print("Skeleton shape:", sample["skeleton_mat"].shape)
print("Label:", sample["label"])
print("Subject:", sample["subject"])


Sample keys: dict_keys(['depth_mat', 'sensor_mat', 'skeleton_mat', 'label', 'subject', 'basename'])
Depth shape: (60, 424, 512)
Sensor shape: (10, 1)
Skeleton shape: (59, 100)
Label: a13
Subject: cx


## Simulate Raw Data Corruption
Apply corruption to raw sensor signals **before** feature extraction to realistically simulate faulty sensors.

In [4]:
import copy

def safe_to_float_array(arr):
    """Safely convert potentially jagged/object arrays to float."""
    try:
        return np.array(arr, dtype=float)
    except (ValueError, TypeError):
        flat = []
        for val in np.array(arr).flat:
            if isinstance(val, np.ndarray):
                flat.extend(val.flatten().tolist())
            else:
                flat.append(float(val))
        return np.array(flat, dtype=float)


def apply_gaussian_noise_inplace(data, noise_fraction=0.10, seed=42):
    """
    Add structured Gaussian noise to a random fraction of timesteps
    in sensor & skeleton data. Corrupts entire frames at once to
    simulate realistic sensor-failure patterns.
    Depth is skipped (camera modality, too large for in-memory noise).
    """
    rng = np.random.RandomState(seed)

    for sample in tqdm(data, desc=f"Applying structured Gaussian noise to {noise_fraction*100:.0f}% of frames"):
        for key in ["sensor_mat", "skeleton_mat"]:
            arr = sample[key]

            if arr.dtype.kind == 'O' or arr.dtype.kind not in ('f',):
                try:
                    arr = np.array(arr, dtype=np.float64)
                except (ValueError, TypeError):
                    arr = safe_to_float_array(arr)
                sample[key] = arr

            signal_std = arr.std()
            if signal_std == 0:
                signal_std = 1.0

            # --- Structured corruption ---
            if arr.ndim >= 2:
                n_frames = arr.shape[0]
                n_corrupt_frames = max(1, int(noise_fraction * n_frames))
                corrupt_indices = rng.choice(
                    n_frames,
                    size=n_corrupt_frames,
                    replace=False
                )
                noise = rng.normal(
                    0,
                    0.2 * signal_std,
                    size=arr[corrupt_indices].shape
                )
                arr[corrupt_indices] += noise
            else:
                # Fallback for 1D signals
                mask = rng.random(arr.shape) < noise_fraction
                n_corrupted = mask.sum()
                if n_corrupted > 0:
                    arr[mask] += rng.normal(0, 0.2 * signal_std, n_corrupted)

    print(f"Structured Gaussian noise applied in-place to sensor & skeleton: {noise_fraction*100:.0f}% of frames")

In [6]:
apply_gaussian_noise_inplace(X, noise_fraction=0.10)
print(f"Done — {len(X)} samples corrupted")

Applying structured Gaussian noise to 10% of frames: 100%|██████████| 1165/1165 [00:06<00:00, 184.75it/s]

Structured Gaussian noise applied in-place to sensor & skeleton: 10% of frames
Done — 1165 samples corrupted


## Feature Extraction

In [5]:
def extract_features_from_depth(depth):
    """
    depth: np.array of shape (frames, H, W) for CZU-MHAD
    If shape is (H, W, frames), we transpose first (UTD-MHAD style).
    Returns: 1D feature vector (37 features)
    """
    depth = np.array(depth)

    # Handle both (H, W, frames) and (frames, H, W) layouts
    # If last dim is small relative to first two, assume (H, W, frames) and transpose
    if depth.ndim == 3 and depth.shape[2] < depth.shape[0] and depth.shape[2] < depth.shape[1]:
        depth = np.transpose(depth, (2, 0, 1))  # (frames, H, W)

    features = []
    flattened = depth.reshape(depth.shape[0], -1)  # (frames, H*W)

    # Temporal stats per frame
    frame_mean = flattened.mean(axis=1)
    frame_std = flattened.std(axis=1)
    frame_min = flattened.min(axis=1)
    frame_max = flattened.max(axis=1)

    # Aggregate over time
    features.extend([frame_mean.mean(), frame_mean.std()])
    features.extend([frame_std.mean(), frame_std.std()])
    features.extend([frame_min.mean(), frame_min.std()])
    features.extend([frame_max.mean(), frame_max.std()])

    # Histogram features over all pixels
    hist, _ = np.histogram(flattened, bins=20)
    features.extend(hist / hist.sum())  # normalize

    # Percentiles over all pixel intensities and frames
    features.extend(np.percentile(flattened, [10, 25, 50, 75, 90]))

    # Temporal differences (motion)
    diff = np.diff(flattened, axis=0)
    features.extend([diff.mean(), diff.std(), diff.max(), diff.min()])

    return np.array(features)

In [6]:
def flatten_numeric(x):
    flat = []
    if isinstance(x, (list, tuple, np.ndarray)):
        for item in x:
            flat.extend(flatten_numeric(item))
    else:
        flat.append(float(x))
    return flat

In [7]:
def extract_features_from_sensor(sensor):
    """
    Enriched sensor feature extraction for CZU-MHAD.
    sensor shape: (time_steps, channels) — e.g. (10, 1)
    Extracts ~90 features: per-channel stats + temporal + frequency + higher-order.
    """
    # Handle jagged/nested sensor data
    try:
        sensor = np.array(sensor, dtype=float)
    except (ValueError, TypeError):
        all_values = flatten_numeric(sensor)
        sensor = np.array(all_values, dtype=float).reshape(-1, 1)

    if sensor.ndim == 1:
        sensor = sensor.reshape(-1, 1)
    if sensor.ndim > 2:
        sensor = sensor.reshape(sensor.shape[0], -1)

    features = []
    n_channels = sensor.shape[1]

    for ch in range(n_channels):
        cd = sensor[:, ch]  # channel data

        # === 1. Basic stats (9) — original ===
        features.append(cd.mean())
        features.append(cd.std())
        features.append(cd.min())
        features.append(cd.max())
        features.extend(np.percentile(cd, [10, 25, 50, 75, 90]))

        # === 2. Additional stats (7) ===
        features.append(cd.max() - cd.min())                    # range
        features.append(np.percentile(cd, 75) - np.percentile(cd, 25))  # IQR
        features.append(np.sqrt(np.mean(cd**2)))                # RMS
        features.append(np.sum(cd**2))                          # energy
        from scipy.stats import skew, kurtosis
        features.append(skew(cd))
        features.append(kurtosis(cd))
        features.append(np.mean(np.abs(cd)))                    # mean absolute value

        # === 3. Temporal / difference features (10) ===
        if len(cd) > 1:
            diff1 = np.diff(cd)
            features.append(diff1.mean())
            features.append(diff1.std())
            features.append(diff1.min())
            features.append(diff1.max())
            features.append(np.mean(np.abs(diff1)))             # mean abs diff
        else:
            features.extend([0]*5)

        if len(cd) > 2:
            diff2 = np.diff(cd, n=2)
            features.append(diff2.mean())
            features.append(diff2.std())
            features.append(np.mean(np.abs(diff2)))
        else:
            features.extend([0]*3)

        # Zero-crossing rate
        if len(cd) > 1:
            zero_crossings = np.sum(np.diff(np.sign(cd - cd.mean())) != 0)
            features.append(zero_crossings / len(cd))
        else:
            features.append(0)

        # Slope (linear trend)
        features.append(np.polyfit(np.arange(len(cd)), cd, 1)[0] if len(cd) > 1 else 0)

        # === 4. Frequency domain features (10) ===
        fft_vals = np.abs(np.fft.rfft(cd))
        n_fft = len(fft_vals)

        # Top-5 FFT magnitudes (padded if needed)
        top_k = 5
        if n_fft >= top_k:
            features.extend(np.sort(fft_vals)[::-1][:top_k])
        else:
            features.extend(list(fft_vals) + [0]*(top_k - n_fft))

        # Spectral stats
        features.append(fft_vals.mean())                        # spectral mean
        features.append(fft_vals.std())                         # spectral std
        features.append(np.sum(fft_vals**2))                    # spectral energy

        # Spectral centroid
        freqs = np.arange(n_fft)
        if fft_vals.sum() > 0:
            features.append(np.sum(freqs * fft_vals) / fft_vals.sum())
        else:
            features.append(0)

        # Dominant frequency index
        features.append(np.argmax(fft_vals) if n_fft > 0 else 0)

        # === 5. Autocorrelation features (5) ===
        if len(cd) > 3:
            autocorr = np.correlate(cd - cd.mean(), cd - cd.mean(), mode='full')
            autocorr = autocorr[len(autocorr)//2:]
            if autocorr[0] != 0:
                autocorr = autocorr / autocorr[0]
            for lag in [1, 2, 3]:
                features.append(autocorr[lag] if lag < len(autocorr) else 0)
        else:
            features.extend([0]*3)

        # Autocorrelation energy and peak
        if len(cd) > 3 and autocorr[0] != 0:
            features.append(np.sum(autocorr[:min(5, len(autocorr))]**2))
            features.append(np.argmax(autocorr[1:min(5, len(autocorr))]) + 1 if len(autocorr) > 1 else 0)
        else:
            features.extend([0, 0])

        # === 6. Signal shape features (5) ===
        features.append(np.argmax(cd) / max(len(cd)-1, 1))     # relative position of max
        features.append(np.argmin(cd) / max(len(cd)-1, 1))     # relative position of min
        features.append(cd[-1] - cd[0] if len(cd) > 1 else 0)  # start-to-end change
        # Coefficient of variation
        features.append(cd.std() / cd.mean() if cd.mean() != 0 else 0)
        # Signal-to-noise ratio (mean/std)
        features.append(cd.mean() / cd.std() if cd.std() != 0 else 0)

    # Total per channel: 9 + 7 + 10 + 10 + 5 + 5 = 46
    # For the full signal (across all channels combined):
    flat = sensor.flatten()

    # === 7. Global cross-channel features (if multi-channel) or global signal features ===
    features.append(flat.mean())
    features.append(flat.std())
    features.append(flat.min())
    features.append(flat.max())
    features.extend(np.percentile(flat, [5, 25, 50, 75, 95]))

    # Global FFT
    global_fft = np.abs(np.fft.rfft(flat))
    top_k_global = 5
    if len(global_fft) >= top_k_global:
        features.extend(np.sort(global_fft)[::-1][:top_k_global])
    else:
        features.extend(list(global_fft) + [0]*(top_k_global - len(global_fft)))

    # Global temporal
    if len(flat) > 1:
        gdiff = np.diff(flat)
        features.extend([gdiff.mean(), gdiff.std(), gdiff.min(), gdiff.max(),
                         np.mean(np.abs(gdiff))])
    else:
        features.extend([0]*5)

    # Pad/truncate raw values to fixed length (10 values)
    raw_padded = np.zeros(10)
    raw_vals = flat[:10]
    raw_padded[:len(raw_vals)] = raw_vals
    features.extend(raw_padded)

    # Total: 46*n_channels + 9 + 5 + 5 + 10 = 46*1 + 29 = ~75 for 1-channel
    # Plus 10 raw = ~85. Close to 90.

    return np.array(features, dtype=float)

In [8]:
def extract_features_from_skeleton(skeleton):
    """
    UTD-MHAD style skeleton feature extraction.
    Handles multiple skeleton shapes:
      - (joints, coords, frames): UTD-MHAD style -> transpose to (frames, joints*coords)
      - (frames, features): already 2D
      - (frames, joints, coords): reshape to (frames, joints*coords)
    Returns: 1D feature vector (4 * num_joint_features)
    """
    skeleton = np.array(skeleton)

    if skeleton.ndim == 3:
        # Determine axis ordering
        # If shape is (joints, coords, frames) where joints~20, coords~3, frames~50+
        if skeleton.shape[1] <= 4 and skeleton.shape[2] > skeleton.shape[0]:
            # (joints, coords, frames) — UTD style
            reshaped = skeleton.transpose(2, 0, 1).reshape(skeleton.shape[2], -1)
        elif skeleton.shape[2] <= 4:
            # (frames, joints, coords)
            reshaped = skeleton.reshape(skeleton.shape[0], -1)
        else:
            # (joints, coords, frames) general case
            reshaped = skeleton.transpose(2, 0, 1).reshape(skeleton.shape[2], -1)
    elif skeleton.ndim == 2:
        reshaped = skeleton
    else:
        reshaped = skeleton.reshape(-1, 1)

    features = []
    features.extend(reshaped.mean(axis=0))   # mean per feature across frames
    features.extend(reshaped.std(axis=0))
    features.extend(reshaped.min(axis=0))
    features.extend(reshaped.max(axis=0))

    return np.array(features)

In [11]:
X_feat, y, subjects = [], [], []

for sample in tqdm(X, desc="Extracting features"):
    depth_feat = extract_features_from_depth(sample["depth_mat"])
    sensor_feat = extract_features_from_sensor(sample["sensor_mat"])
    skeleton_feat = extract_features_from_skeleton(sample["skeleton_mat"])

    X_feat.append({
        "depth_feat": depth_feat,
        "sensor_feat": sensor_feat,
        "skeleton_feat": skeleton_feat
    })
    y.append(sample["label"])
    subjects.append(sample["subject"])

print("\nFeature extraction complete!")
print("Depth Features:", X_feat[0]["depth_feat"].shape)
print("Sensor Features:", X_feat[0]["sensor_feat"].shape)
print("Skeleton Features:", X_feat[0]["skeleton_feat"].shape)
print("\nDataset size:", len(X_feat))

Extracting features: 100%|██████████| 1165/1165 [24:24<00:00,  1.26s/it]


Feature extraction complete!
Depth Features: (37,)
Sensor Features: (75,)
Skeleton Features: (400,)

Dataset size: 1165


In [12]:
le = LabelEncoder()
le.fit(y)
y_enc = le.transform(y)
print("\nEncoded:", y_enc.shape)
print("Classes:", le.classes_)


Encoded: (1165,)
Classes: ['a1' 'a10' 'a11' 'a12' 'a13' 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a2'
 'a20' 'a21' 'a22' 'a3' 'a4' 'a5' 'a6' 'a7' 'a8' 'a9']


In [13]:
import joblib

SAVE_DIR = "features_10_noise"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(X_feat, os.path.join(SAVE_DIR, "X_feat.pkl"), compress=3)
print("\nSaved feature dictionary")

np.save(os.path.join(SAVE_DIR, "y.npy"), y_enc)
print("Saved encoded labels")

np.save(os.path.join(SAVE_DIR, "subjects.npy"), subjects)
print("Saved subjects")

joblib.dump(le, os.path.join(SAVE_DIR, "label_encoder.pkl"))
print("Saved LabelEncoder")


Saved feature dictionary
Saved encoded labels
Saved subjects
Saved LabelEncoder


In [14]:
import joblib

X_feat = joblib.load("features_10_noise/X_feat.pkl")
y = np.load("features_10_noise/y.npy")
subjects = np.load("features_10_noise/subjects.npy")

le = joblib.load("features_10_noise/label_encoder.pkl")
print("Classes:", le.classes_)

Classes: ['a1' 'a10' 'a11' 'a12' 'a13' 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a2'
 'a20' 'a21' 'a22' 'a3' 'a4' 'a5' 'a6' 'a7' 'a8' 'a9']


In [15]:
print("Samples:", len(X_feat))
print("Depth feat shape:", X_feat[0]["depth_feat"].shape)
print("Sensor feat shape:", X_feat[0]["sensor_feat"].shape)
print("Skeleton feat shape:", X_feat[0]["skeleton_feat"].shape)

Samples: 1165
Depth feat shape: (37,)
Sensor feat shape: (75,)
Skeleton feat shape: (400,)


## Data Preprocessing

In [16]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.preprocessing import StandardScaler
import numpy as np

X_depth = np.stack([x["depth_feat"] for x in X_feat])
X_sensor = np.stack([x["sensor_feat"] for x in X_feat])
X_skel = np.stack([x["skeleton_feat"] for x in X_feat])

scaler_depth = StandardScaler()
X_depth = scaler_depth.fit_transform(X_depth)

scaler_sensor = StandardScaler()
X_sensor = scaler_sensor.fit_transform(X_sensor)

scaler_skel = StandardScaler()
X_skel = scaler_skel.fit_transform(X_skel)

# Store dimensions for model definitions
DEPTH_DIM = X_depth.shape[1]
SENSOR_DIM = X_sensor.shape[1]
SKEL_DIM = X_skel.shape[1]

print("Feature shapes:")
print(f"Depth: {X_depth.shape}  (dim={DEPTH_DIM})")
print(f"Sensor: {X_sensor.shape}  (dim={SENSOR_DIM})")
print(f"Skeleton: {X_skel.shape}  (dim={SKEL_DIM})")

Feature shapes:
Depth: (1165, 37)  (dim=37)
Sensor: (1165, 75)  (dim=75)
Skeleton: (1165, 400)  (dim=400)


## Random 25% noise

In [ ]:
X = load_czu_mhad_dataset()

Subjects: ['cx', 'cyy', 'myj', 'qyh', 'zyh']
Total samples loaded metadata: 1165


Loading all samples:  86%|████████▌ | 998/1165 [00:57<00:17,  9.81it/s]

In [9]:
apply_gaussian_noise_inplace(X, noise_fraction=0.25)
print(f"Done — {len(X)} samples corrupted")

Applying structured Gaussian noise to 25% of frames: 100%|██████████| 1165/1165 [00:07<00:00, 163.43it/s]

Structured Gaussian noise applied in-place to sensor & skeleton: 25% of frames
Done — 1165 samples corrupted


In [10]:
X_feat, y, subjects = [], [], []

for sample in tqdm(X, desc="Extracting features"):
    depth_feat = extract_features_from_depth(sample["depth_mat"])
    sensor_feat = extract_features_from_sensor(sample["sensor_mat"])
    skeleton_feat = extract_features_from_skeleton(sample["skeleton_mat"])

    X_feat.append({
        "depth_feat": depth_feat,
        "sensor_feat": sensor_feat,
        "skeleton_feat": skeleton_feat
    })
    y.append(sample["label"])
    subjects.append(sample["subject"])

print("\nFeature extraction complete!")
print("Depth Features:", X_feat[0]["depth_feat"].shape)
print("Sensor Features:", X_feat[0]["sensor_feat"].shape)
print("Skeleton Features:", X_feat[0]["skeleton_feat"].shape)
print("\nDataset size:", len(X_feat))

Extracting features: 100%|██████████| 1165/1165 [25:39<00:00,  1.32s/it]


Feature extraction complete!
Depth Features: (37,)
Sensor Features: (75,)
Skeleton Features: (400,)

Dataset size: 1165


In [11]:
le = LabelEncoder()
le.fit(y)
y_enc = le.transform(y)
print("\nEncoded:", y_enc.shape)
print("Classes:", le.classes_)


Encoded: (1165,)
Classes: ['a1' 'a10' 'a11' 'a12' 'a13' 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a2'
 'a20' 'a21' 'a22' 'a3' 'a4' 'a5' 'a6' 'a7' 'a8' 'a9']


In [12]:
import joblib

SAVE_DIR = "features_25_noise"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(X_feat, os.path.join(SAVE_DIR, "X_feat.pkl"), compress=3)
print("\nSaved feature dictionary")

np.save(os.path.join(SAVE_DIR, "y.npy"), y_enc)
print("Saved encoded labels")

np.save(os.path.join(SAVE_DIR, "subjects.npy"), subjects)
print("Saved subjects")

joblib.dump(le, os.path.join(SAVE_DIR, "label_encoder.pkl"))
print("Saved LabelEncoder")


Saved feature dictionary
Saved encoded labels
Saved subjects
Saved LabelEncoder


In [13]:
import joblib

X_feat = joblib.load("features_25_noise/X_feat.pkl")
y = np.load("features_25_noise/y.npy")
subjects = np.load("features_25_noise/subjects.npy")

le = joblib.load("features_25_noise/label_encoder.pkl")
print("Classes:", le.classes_)

Classes: ['a1' 'a10' 'a11' 'a12' 'a13' 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a2'
 'a20' 'a21' 'a22' 'a3' 'a4' 'a5' 'a6' 'a7' 'a8' 'a9']


In [14]:
print("Samples:", len(X_feat))
print("Depth feat shape:", X_feat[0]["depth_feat"].shape)
print("Sensor feat shape:", X_feat[0]["sensor_feat"].shape)
print("Skeleton feat shape:", X_feat[0]["skeleton_feat"].shape)

Samples: 1165
Depth feat shape: (37,)
Sensor feat shape: (75,)
Skeleton feat shape: (400,)


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.preprocessing import StandardScaler
import numpy as np

X_depth = np.stack([x["depth_feat"] for x in X_feat])
X_sensor = np.stack([x["sensor_feat"] for x in X_feat])
X_skel = np.stack([x["skeleton_feat"] for x in X_feat])

scaler_depth = StandardScaler()
X_depth = scaler_depth.fit_transform(X_depth)

scaler_sensor = StandardScaler()
X_sensor = scaler_sensor.fit_transform(X_sensor)

scaler_skel = StandardScaler()
X_skel = scaler_skel.fit_transform(X_skel)

# Store dimensions for model definitions
DEPTH_DIM = X_depth.shape[1]
SENSOR_DIM = X_sensor.shape[1]
SKEL_DIM = X_skel.shape[1]

print("Feature shapes:")
print(f"Depth: {X_depth.shape}  (dim={DEPTH_DIM})")
print(f"Sensor: {X_sensor.shape}  (dim={SENSOR_DIM})")
print(f"Skeleton: {X_skel.shape}  (dim={SKEL_DIM})")

Feature shapes:
Depth: (1165, 37)  (dim=37)
Sensor: (1165, 75)  (dim=75)
Skeleton: (1165, 400)  (dim=400)
